Problem 1: Trip duration
 Part 1: Build a Regression Model
Build a regression to predict trip duration by using

Day of time
Distance between start and end stations (there might be more than one way to measure it)
Hour of day
Weekend indicator
Don't forget to model bias (this one is intentionally not used in lecture)
Also any thing you want to end
 Part 2: Experiment Design
Ensure that you properly design your experiment to report unbiased performance metric you choose
 Part 3 [Optional]: Visualize
Generate some fictional pickup and dropoff locations for bike trips (random pair selection)
Estimate trip duration for those say 10 trips
Visualize them on map using pydeck by using redish color for slower trips and greener for faster trips.

In [1]:
!pip install duckdb
%load_ext duckdb

The duckdb module is not an IPython extension.


In [1]:
%%duckdb


SELECT 
  *, date_diff('minute', start_at, stop_at) AS duration_min
FROM (
  SELECT 
    * EXCLUDE (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
)
LIMIT 3

,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender,start_at,stop_at,duration_min
0,455,1 Ave & E 44 St,40.750020,-73.969053,265,Stanton St & Chrystie St,40.722293,-73.991475,18660,Subscriber,1960.0,2,2015-01-01 00:01:00,2015-01-01 00:24:00,23
1,434,9 Ave & W 18 St,40.743174,-74.003664,482,W 15 St & 7 Ave,40.739355,-73.999318,16085,Subscriber,1963.0,1,2015-01-01 00:02:00,2015-01-01 00:08:00,6
2,491,E 24 St & Park Ave S,40.740964,-73.986022,505,6 Ave & W 33 St,40.749013,-73.988484,20845,Subscriber,1974.0,1,2015-01-01 00:04:00,2015-01-01 00:10:00,6


In [2]:
%%duckdb

SET memory_limit='6GB'; 
SET threads=2;

CREATE OR REPLACE TABLE raw_trips AS
SELECT 
    "start station latitude", "start station longitude",
    "end station latitude", "end station longitude",
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet';

CREATE OR REPLACE TABLE trips AS
SELECT 
    *,
    sqrt(power("end station latitude" - "start station latitude", 2) + 
         power("end station longitude" - "start station longitude", 2)) AS distance_approx,
    hour(start_at) AS hour_of_day,
    CASE 
        WHEN hour(start_at) BETWEEN 6 AND 11 THEN 'morning'
        WHEN hour(start_at) BETWEEN 12 AND 17 THEN 'afternoon'
        WHEN hour(start_at) BETWEEN 18 AND 21 THEN 'evening'
        ELSE 'night'
    END AS time_of_day,
    CASE WHEN dayofweek(start_at) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend,
    1 AS bias_term,
    date_diff('minute', start_at, stop_at) AS duration_min
FROM raw_trips
WHERE date_diff('minute', start_at, stop_at) BETWEEN 1 AND 180;

DROP TABLE raw_trips;

,Success


In [4]:

df_check = duckdb.query("SELECT * FROM trips LIMIT 5").df()
display(df_check)

,start station latitude,start station longitude,end station latitude,end station longitude,start_at,stop_at,distance_approx,hour_of_day,time_of_day,is_weekend,bias_term,duration_min
0,40.750020,-73.969053,40.722293,-73.991475,2015-01-01 00:01:00,2015-01-01 00:24:00,0.035658,0,night,0,1,23
1,40.743174,-74.003664,40.739355,-73.999318,2015-01-01 00:02:00,2015-01-01 00:08:00,0.005786,0,night,0,1,6
2,40.740964,-73.986022,40.749013,-73.988484,2015-01-01 00:04:00,2015-01-01 00:10:00,0.008417,0,night,0,1,6
3,40.683178,-73.965964,40.688515,-73.964763,2015-01-01 00:04:00,2015-01-01 00:07:00,0.005471,0,night,0,1,3
4,40.745168,-73.986831,40.726218,-73.983799,2015-01-01 00:05:00,2015-01-01 00:21:00,0.019191,0,night,0,1,16


In [5]:
%%duckdb -o df_sampled
SELECT * FROM trips 
USING SAMPLE 300000;

,start station latitude,start station longitude,end station latitude,end station longitude,start_at,stop_at,distance_approx,hour_of_day,time_of_day,is_weekend,bias_term,duration_min
0,40.767272,-73.993929,40.757246,-73.978059,2016-10-19 13:05:23,2016-10-19 13:28:30,0.018772,13,afternoon,0,1,23
1,40.694281,-73.992300,40.714948,-74.002345,2016-12-13 09:16:27,2016-12-13 09:30:19,0.022979,9,morning,0,1,14
2,40.758491,-73.959206,40.765265,-73.981923,2015-03-19 19:00:00,2015-03-19 19:13:00,0.023706,19,evening,0,1,13
3,40.711512,-74.015756,40.760301,-73.998842,2015-09-18 17:36:00,2015-09-18 17:55:11,0.051638,17,afternoon,0,1,19
4,40.729554,-73.980572,40.731724,-74.006744,2016-11-18 09:17:38,2016-11-18 09:32:30,0.026262,9,morning,0,1,15
...,...,...,...,...,...,...,...,...,...,...,...,...
299995,40.760958,-73.967245,40.757973,-73.966033,2015-06-15 09:46:00,2015-06-15 09:49:00,0.003221,9,morning,0,1,3
299996,40.762288,-73.983362,40.751581,-73.977910,2016-05-26 08:00:36,2016-05-26 08:09:24,0.012015,8,morning,0,1,9
299997,40.764397,-73.973715,40.749370,-73.999234,2016-12-03 16:54:15,2016-12-03 17:20:08,0.029615,16,afternoon,1,1,26
299998,40.700379,-73.995481,40.695128,-73.995951,2016-08-31 07:56:10,2016-08-31 08:00:58,0.005271,7,morning,0,1,4


In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

features = ["distance_approx", "time_of_day", "is_weekend", "bias_term"]
X = df_sampled[features]
X_new = pd.get_dummies(X, columns=["time_of_day"], drop_first=True)
y = df_sampled["duration_min"]

x_train, x_val, y_train, y_val = train_test_split(X_new, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(x_train, y_train)

y_pred = model.predict(x_val)

print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.2f}")
print(f"R2 Score: {r2_score(y_val, y_pred):.4f}")

RMSE: 12.17
R2 Score: 0.0157


In [7]:
import pydeck as pdk
import numpy as np
import pandas as pd

np.random.seed(42)

df_sample_locs = df_sampled[['start station latitude', 'start station longitude', 
                             'end station latitude', 'end station longitude']].dropna().sample(10)

df_sample_locs['duration_min'] = np.random.uniform(5, 45, 10)

def get_color(duration):
    if duration < 20:
        return [0, 255, 0, 160] 
    else:
        return [255, 0, 0, 160]

df_sample_locs['color'] = df_sample_locs['duration_min'].apply(get_color)

view_state = pdk.ViewState(latitude=40.73, longitude=-74.00, zoom=12)

layer = pdk.Layer(
    "ArcLayer",
    df_sample_locs,
    get_source_position=["start station longitude", "start station latitude"],
    get_target_position=["end station longitude", "end station latitude"],
    get_source_color="color",
    get_target_color="color",
    get_width=4,
    pickable=True,
)

r = pdk.Deck(layers=[layer], initial_view_state=view_state)
r.to_html("trips_viz.html")